# Process Simulation Outputs — Python version

**Direct Python mirror of `03_Process_Sims_Outputs.qmd`.**  
Every section matches the R script exactly — same logic, same calculations, same outputs.  
Use this notebook to understand what the R script does and to experiment with results.

**What this notebook does (in order):**
1. Load observed PK data (monkey + human)
2. Load simulation CSVs from the latest run folder
3. Convert units (µmol/L → µg/mL using MW)
4. Summarise simulations (5th / 50th / 95th percentile per timepoint)
5. Summarise observed data (mean ± SD per timepoint)
6. Plot observed vs predicted PK profiles (log-scale, 28-day window)
7. Compute Cmax and AUC₀₋₆₇₂ₕ using log-trapezoidal rule
8. Compute fold errors (FE = Predicted / Observed)
9. Display colour-coded tables (green = within 2-fold)
10. Export metrics to Excel

## 0 — Install / import libraries

Run `pip install pandas numpy matplotlib seaborn openpyxl` once if not already installed.

In [ ]:
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.cm import get_cmap
import seaborn as sns
from IPython.display import display

# ── Paths (same relative structure as the R script) ──────────────────────────
# This notebook lives in: Test2/03_Apriori/V2/Scripts/
SCRIPT_DIR  = os.path.dirname(os.path.abspath('__file__'))   # Scripts/
DATA_DIR    = os.path.normpath(os.path.join(SCRIPT_DIR, '../../../01_Data'))
RESULTS_DIR = os.path.normpath(os.path.join(SCRIPT_DIR, '../SimsOutputs/SimulationResults'))
FIGURES_DIR = os.path.normpath(os.path.join(SCRIPT_DIR, '../SimsOutputs/Figures'))

AUC_END_H = 28 * 24   # 672 h — 28-day analysis window

print('Data dir   :', DATA_DIR)
print('Results dir:', RESULTS_DIR)
print('Figures dir:', FIGURES_DIR)

## 1 — Helper functions

These are direct translations of the R helper functions defined at the top of the R script.

In [ ]:
def normalize_species(s):
    """
    R: normalize_species()
    Standardises species label strings so they match simulation scenario names.
    """
    s = str(s).strip().lower()
    if s == 'human':                        return 'Human'
    if s in ('nhp', 'monkey', 'primate'):   return 'Monkey'
    if s == 'rat':                          return 'Rat'
    if s == 'mouse':                        return 'Mouse'
    if s == 'dog':                          return 'Dog'
    return s.title()


def parse_scenario(filepath):
    """
    R: parse_scenario()
    Extracts Molecule, Species, and Dose from a simulation CSV filename.
    Example: 'Pembrolizumab_Human_10_mpk.csv' → ('Pembrolizumab', 'Human', 10.0)
    """
    base   = os.path.basename(filepath)
    base   = re.sub(r'\.csv$', '', base)
    base   = re.sub(r'_mpk$', '', base)
    parts  = base.split('_')
    dose   = float(parts[-1])
    species = parts[-2]
    molecule = '_'.join(parts[:-2])
    return molecule, species, dose


def trapz_auc(time, conc, t_end=np.inf):
    """
    R: trapz_auc()
    Log-linear trapezoidal AUC integration.
    - Uses LOG trapezoid on strictly declining segments (more accurate for PK)
    - Falls back to LINEAR trapezoid when concentrations are equal or zero
    - Only integrates up to t_end (672 h = 28 days here)
    """
    mask = (~np.isnan(time)) & (~np.isnan(conc)) & (conc > 0) & (time <= t_end)
    t = time[mask]
    c = conc[mask]
    if len(t) < 2:
        return np.nan

    dt = np.diff(t)
    c1 = c[:-1]
    c2 = c[1:]

    use_log = (c1 != c2) & (c1 > 0) & (c2 > 0)
    trap = np.where(
        use_log,
        dt * (c1 - c2) / np.log(c1 / c2),   # log-trapezoidal
        dt * (c1 + c2) / 2                   # linear trapezoidal
    )
    return float(np.sum(trap))


print('Helper functions defined.')

## 2 — Load observed data

**R equivalent:** the `load-obs` chunk.  
- Monkey data: `01_Data/Monkey/mAb_data_clean.csv`  
- Human data:  `01_Data/Human/mAb_data_human.csv`  
- Both filtered to 0–672 h (28 days) and quantifiable concentrations only.

In [ ]:
# ── Monkey observed data ──────────────────────────────────────────────────────
obs_raw = pd.read_csv(os.path.join(DATA_DIR, 'Monkey', 'mAb_data_clean.csv'))

obs_raw['Species']   = obs_raw['Species'].apply(normalize_species)
obs_raw['Time_h']    = pd.to_numeric(obs_raw['Time'], errors='coerce')
obs_raw['Conc_ugmL'] = pd.to_numeric(obs_raw['Measurement'], errors='coerce')

obs = (
    obs_raw
    .query('Conc_ugmL > 0 and Conc_ugmL == Conc_ugmL')   # drop NaN and zero
    .query('Time_h <= @AUC_END_H')                         # 28-day window
    .copy()
)

print(f'Monkey obs: {len(obs):,} rows | {obs["Molecule"].nunique()} drugs')

# ── Human observed data ───────────────────────────────────────────────────────
obs_human_raw = pd.read_csv(os.path.join(DATA_DIR, 'Human', 'mAb_data_human.csv'))

obs_human_raw['Species']   = obs_human_raw['Species'].apply(normalize_species)
obs_human_raw['Time_h']    = pd.to_numeric(obs_human_raw['Time'], errors='coerce')
obs_human_raw['Conc_ugmL'] = pd.to_numeric(obs_human_raw['Measurement'], errors='coerce')

obs_human = (
    obs_human_raw
    .query('Conc_ugmL > 0 and Conc_ugmL == Conc_ugmL')
    .query('Time_h <= @AUC_END_H')
    # Exclude SC routes — IV-only PBPK model cannot be compared to SC data
    .loc[lambda d: ~d['Route'].str.strip().str.upper().eq('SC')]
    .copy()
)

print(f'Human obs:  {len(obs_human):,} rows | {obs_human["Molecule"].nunique()} drugs')

# ── MW lookup (used for unit conversion of simulation output) ─────────────────
# µmol/L × MW (g/mol) / 1000 = µg/mL
mw_lookup = (
    pd.concat([
        obs_raw[['Molecule', 'Molecular.Weight']].rename(columns={'Molecular.Weight': 'MW'}),
        obs_human_raw[['Molecule', 'Molecular.Weight']].rename(columns={'Molecular.Weight': 'MW'})
    ])
    .drop_duplicates('Molecule')
    .assign(MW=lambda d: pd.to_numeric(d['MW'], errors='coerce'))
    .set_index('Molecule')['MW']
    .to_dict()
)

print('\nMW lookup (Da):')
for mol, mw in sorted(mw_lookup.items()):
    print(f'  {mol:<20} {mw:,.0f}')

## 3 — Load simulation results

**R equivalent:** the `load-sims` chunk.  
- Auto-selects the most recently modified results folder (same as R's `which.max(file.mtime(...))`)  
- Reads each CSV, extracts the concentration column, converts µmol/L → µg/mL

In [ ]:
# ── Auto-select latest results folder (same logic as R) ───────────────────────
sim_folders = [d for d in glob.glob(os.path.join(RESULTS_DIR, '*')) if os.path.isdir(d)]
if not sim_folders:
    raise FileNotFoundError(f'No results folders found in: {RESULTS_DIR}')

latest_dir = max(sim_folders, key=os.path.getmtime)
print('Results folder:', latest_dir)

# ── List scenario CSVs ────────────────────────────────────────────────────────
csv_files = [
    f for f in glob.glob(os.path.join(latest_dir, '*.csv'))
    if not f.endswith('_population.csv')
]
print(f'{len(csv_files)} scenario CSV files found')


def read_one_csv(filepath):
    """
    R: read_one_csv()
    Reads one simulation CSV and returns a tidy DataFrame with columns:
    Molecule, Species, Dose, IndividualId, Time_h, Conc_ugmL
    
    The CSV columns look like:
      IndividualId | Time [min] | Organism|PeripheralVenousBlood|mAb|Plasma... [µmol/l]
    """
    molecule, species, dose = parse_scenario(filepath)

    try:
        df = pd.read_csv(filepath)
    except Exception:
        return None

    # Find the concentration column (contains 'PeripheralVenousBlood' and 'mAb')
    conc_cols = [c for c in df.columns
                 if 'PeripheralVenousBlood' in c and 'Plasma' in c and '|mAb|' in c]
    if not conc_cols:
        return None
    conc_col = conc_cols[0]

    mw = mw_lookup.get(molecule)
    if mw is None or np.isnan(mw):
        return None

    out = pd.DataFrame({
        'Molecule':     molecule,
        'Species':      species,
        'Dose':         dose,
        'IndividualId': df['IndividualId'].astype(int),
        'Time_h':       df['Time [min]'].astype(float) / 60,   # min → hours
        'Conc_ugmL':    df[conc_col].astype(float) * mw / 1000  # µmol/L → µg/mL
    })
    return out


# Read all CSVs (sequential — Python; R uses parallel cluster)
sim_list = [read_one_csv(f) for f in csv_files]
sim = pd.concat([d for d in sim_list if d is not None], ignore_index=True)

if sim.empty:
    raise ValueError('No concentration data found in any scenario CSV.')

n_scen = sim.groupby(['Molecule', 'Species', 'Dose']).ngroups
print(f'Loaded {n_scen} scenarios | {sim["IndividualId"].nunique()} individual(s) per scenario')
sim.head()

## 4 — Summarise for plotting

**R equivalent:** the `summarise` chunk.

- **Simulated:** 5th / 50th (median) / 95th percentile across virtual individuals at each timepoint → forms the shaded ribbon + centre line
- **Observed:** mean ± SD across subjects at each timepoint → forms the points + error bars

In [ ]:
# ── Simulated: percentile summary per timepoint ───────────────────────────────
sim_summary = (
    sim
    .groupby(['Molecule', 'Species', 'Dose', 'Time_h'])['Conc_ugmL']
    .quantile([0.05, 0.50, 0.95])
    .unstack()
    .reset_index()
    .rename(columns={0.05: 'Pred_p05', 0.50: 'Pred_med', 0.95: 'Pred_p95'})
)

print(f'Simulated summary: {len(sim_summary):,} rows')


def summarise_obs(df):
    """R: summarise_obs() — mean ± SD per Molecule/Species/Dose/Time"""
    grp = df.groupby(['Molecule', 'Species', 'Dose', 'Time_h'])['Conc_ugmL']
    out = grp.agg(
        Obs_mean='mean',
        Obs_sd='std',
        n='count'
    ).reset_index()
    out['Obs_lo'] = (out['Obs_mean'] - out['Obs_sd']).clip(lower=1e-4)
    out['Obs_hi'] =  out['Obs_mean'] + out['Obs_sd']
    return out


obs_summary       = summarise_obs(obs)
obs_human_summary = summarise_obs(obs_human)

# Combined for plotting (both species appear in same figure per drug)
obs_all_summary = pd.concat([obs_summary, obs_human_summary], ignore_index=True)

print(f'Observed summary (monkey): {len(obs_summary):,} rows')
print(f'Observed summary (human):  {len(obs_human_summary):,} rows')

## 5 — PK profiles: observed vs simulated

**R equivalent:** the `pk-plots` chunk.

One figure per drug. Each figure has two panels: Monkey (left) and Human (right).  
- Shaded ribbon = 5th–95th percentile of simulated individuals  
- Solid line = simulated median  
- Points = observed mean  
- Error bars = observed ± 1 SD  
- Y-axis is log₁₀ scale; X-axis clipped at 672 h (28 days)

In [ ]:
os.makedirs(FIGURES_DIR, exist_ok=True)

molecules = sorted(set(sim_summary['Molecule'].unique()) | set(obs_all_summary['Molecule'].unique()))

# Viridis-D colour palette (matches R's scale_colour_viridis_d option='D', end=0.88)
VIRIDIS = plt.cm.get_cmap('viridis')

def dose_colours(doses):
    """Assign one viridis colour per unique dose level."""
    doses_sorted = sorted(doses)
    n = len(doses_sorted)
    return {d: VIRIDIS(i / max(n - 1, 1) * 0.88) for i, d in enumerate(doses_sorted)}


for mol in molecules:
    sim_mol = sim_summary[sim_summary['Molecule'] == mol]
    obs_mol = obs_all_summary[obs_all_summary['Molecule'] == mol]

    all_doses  = sorted(set(sim_mol['Dose'].unique()) | set(obs_mol['Dose'].unique()))
    colours    = dose_colours(all_doses)
    all_species = sorted(set(sim_mol['Species'].unique()) | set(obs_mol['Species'].unique()))

    fig, axes = plt.subplots(1, len(all_species), figsize=(10, 4.5), sharey=False)
    if len(all_species) == 1:
        axes = [axes]

    for ax, species in zip(axes, all_species):
        sim_sp = sim_mol[sim_mol['Species'] == species]
        obs_sp = obs_mol[obs_mol['Species'] == species]

        for dose in all_doses:
            col = colours[dose]
            label = str(dose)

            # Simulated ribbon (5th–95th percentile)
            sd = sim_sp[sim_sp['Dose'] == dose].sort_values('Time_h')
            if not sd.empty:
                ax.fill_between(sd['Time_h'], sd['Pred_p05'], sd['Pred_p95'],
                                color=col, alpha=0.18)
                ax.plot(sd['Time_h'], sd['Pred_med'], color=col,
                        linewidth=1.4, label=label)

            # Observed points + error bars
            od = obs_sp[obs_sp['Dose'] == dose].sort_values('Time_h')
            if not od.empty:
                ax.scatter(od['Time_h'], od['Obs_mean'], color=col,
                           s=25, zorder=5, label=None if not sd.empty else label)
                err_od = od[od['n'] > 1].dropna(subset=['Obs_sd'])
                if not err_od.empty:
                    ax.errorbar(err_od['Time_h'], err_od['Obs_mean'],
                                yerr=err_od['Obs_sd'],
                                fmt='none', color=col, capsize=2, linewidth=0.7)

        ax.set_yscale('log')
        ax.set_xlim(0, 672)              # 28-day window (R: coord_cartesian)
        ax.set_ylim(bottom=0.1)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(
            lambda y, _: f'{y:g}'
        ))
        ax.set_xlabel('Time (h)')
        ax.set_ylabel('Plasma Concentration (µg/mL)')
        ax.set_title(species, fontweight='bold')
        ax.grid(True, which='major', linestyle='--', linewidth=0.4, alpha=0.6)
        ax.grid(False, which='minor')

    # Shared legend (dose colours) below the figure
    handles = [plt.Line2D([0], [0], color=colours[d], linewidth=2, label=str(d))
               for d in all_doses]
    fig.legend(handles, [str(d) for d in all_doses],
               title='Dose (mg/kg)', loc='lower center',
               ncol=min(len(all_doses), 6), frameon=False,
               bbox_to_anchor=(0.5, -0.05))

    fig.suptitle(mol, fontsize=13, fontweight='bold', y=1.01)
    fig.text(0.5, 0.97, 'Lines = simulated  |  Points = observed',
             ha='center', fontsize=9, color='grey')

    plt.tight_layout()
    out_path = os.path.join(FIGURES_DIR, f'PK_ObsPred_{mol}_py.png')
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

## 6 — Compute PK metrics (Cmax and AUC₀₋₆₇₂ₕ)

**R equivalent:** the `pk-metrics` chunk.

**For simulated data:**  
Per individual → AUC (log-trapezoidal) and Cmax → then average across individuals per scenario.

**For observed data:**  
Per subject → AUC and Cmax → then average across subjects per dose group.

In [ ]:
# ── Simulated metrics ─────────────────────────────────────────────────────────
sim_filtered = sim[sim['Time_h'] <= AUC_END_H].copy()
sim_filtered = sim_filtered.sort_values(['Molecule', 'Species', 'Dose', 'IndividualId', 'Time_h'])

def sim_metrics_per_individual(grp):
    t = grp['Time_h'].values
    c = grp['Conc_ugmL'].values
    return pd.Series({
        'Pred_AUC':  trapz_auc(t, c, t_end=AUC_END_H),
        'Pred_Cmax': float(np.max(c)) if len(c) > 0 else np.nan
    })

sim_ind = (
    sim_filtered
    .groupby(['Molecule', 'Species', 'Dose', 'IndividualId'])
    .apply(sim_metrics_per_individual)
    .reset_index()
)

# Average across individuals per scenario
sim_metrics = (
    sim_ind
    .groupby(['Molecule', 'Species', 'Dose'])[['Pred_AUC', 'Pred_Cmax']]
    .mean()
    .reset_index()
)

print(f'Simulated metrics: {len(sim_metrics)} scenarios')


# ── Observed metrics ──────────────────────────────────────────────────────────
def compute_obs_metrics(df):
    """R: compute_obs_metrics() — per subject → mean across subjects"""
    def per_subject(grp):
        t = grp['Time_h'].values
        c = grp['Conc_ugmL'].values
        return pd.Series({
            'Obs_AUC':  trapz_auc(t, c, t_end=AUC_END_H),
            'Obs_Cmax': float(np.max(c)) if len(c) > 0 else np.nan
        })

    per_subj = (
        df
        .groupby(['Molecule', 'Species', 'Dose', 'Subject.Id'])
        .apply(per_subject)
        .reset_index()
    )
    return (
        per_subj
        .groupby(['Molecule', 'Species', 'Dose'])[['Obs_AUC', 'Obs_Cmax']]
        .mean()
        .reset_index()
    )


obs_metrics       = compute_obs_metrics(obs)
obs_human_metrics = compute_obs_metrics(obs_human)

print(f'Observed metrics (monkey): {len(obs_metrics)} dose groups')
print(f'Observed metrics (human):  {len(obs_human_metrics)} dose groups')

## 7 — Compute fold errors

**R equivalent:** `compute_fe()`

**FE = Predicted / Observed**  
- FE within 0.5–2.0 = within 2-fold = acceptable (green)  
- FE outside that range = outside 2-fold = needs attention (red)

In [ ]:
def compute_fe(obs_m, sim_m):
    """R: compute_fe() — outer join observed + simulated, compute FE"""
    merged = obs_m.merge(sim_m, on=['Molecule', 'Species', 'Dose'], how='outer')
    merged['FE_Cmax'] = (merged['Pred_Cmax'] / merged['Obs_Cmax']).round(2)
    merged['FE_AUC']  = (merged['Pred_AUC']  / merged['Obs_AUC'] ).round(2)
    return merged.sort_values(['Molecule', 'Dose'])


sim_metrics_monkey = sim_metrics[sim_metrics['Species'] == 'Monkey']
sim_metrics_human  = sim_metrics[sim_metrics['Species'] == 'Human']

metrics_tbl       = compute_fe(obs_metrics,       sim_metrics_monkey)
human_metrics_tbl = compute_fe(obs_human_metrics, sim_metrics_human)

print('Fold error tables computed.')

## 8 — Display colour-coded fold error tables

**R equivalent:** `render_metrics_kbl()` using kableExtra.  
Green = within 2-fold (0.5–2.0), Red = outside 2-fold.

In [ ]:
def style_fe(val):
    """Colour a fold-error cell green (within 2-fold) or red (outside)."""
    if pd.isna(val):
        return ''
    if 0.5 <= val <= 2.0:
        return 'background-color: #d4edda; color: #155724; font-weight: bold'
    return 'background-color: #f8d7da; color: #721c24; font-weight: bold'


def display_metrics(df, title):
    display_df = df[[
        'Molecule', 'Dose',
        'Obs_Cmax', 'Pred_Cmax', 'FE_Cmax',
        'Obs_AUC',  'Pred_AUC',  'FE_AUC'
    ]].copy()
    display_df.columns = [
        'Molecule', 'Dose (mg/kg)',
        'Obs Cmax (µg/mL)', 'Pred Cmax (µg/mL)', 'FE Cmax',
        'Obs AUC (µg·h/mL)', 'Pred AUC (µg·h/mL)', 'FE AUC'
    ]
    for col in ['Obs Cmax (µg/mL)', 'Pred Cmax (µg/mL)']:
        display_df[col] = display_df[col].round(1)
    for col in ['Obs AUC (µg·h/mL)', 'Pred AUC (µg·h/mL)']:
        display_df[col] = display_df[col].round(0)

    print(f'\n{title}')
    styled = (
        display_df.style
        .applymap(style_fe, subset=['FE Cmax', 'FE AUC'])
        .format(na_rep='—')
        .set_caption(title)
    )
    display(styled)


display_metrics(
    metrics_tbl,
    f'Monkey PK Metrics — AUC integrated over 0–{AUC_END_H} h (28 days). FE = Pred / Obs'
)

human_obs_only = human_metrics_tbl.dropna(subset=['Obs_Cmax', 'Pred_Cmax'])
if not human_obs_only.empty:
    display_metrics(
        human_obs_only,
        f'Human PK Metrics (with observed data) — AUC over 0–{AUC_END_H} h. FE = Pred / Obs'
    )

human_pred_only = human_metrics_tbl[human_metrics_tbl['Obs_Cmax'].isna()]
if not human_pred_only.empty:
    print('\nHuman Predicted-Only — No observed data at these doses')
    display(human_pred_only[['Molecule', 'Dose', 'Pred_Cmax', 'Pred_AUC']].round(1))

## 9 — Export to Excel

**R equivalent:** the `save-metrics` chunk.  
Saves the combined monkey + human fold-error table to `SimsOutputs/Figures/PK_Metrics_ObsPred_py.xlsx`.

Note: this writes a separate `_py.xlsx` file so it doesn't overwrite the R output.

In [ ]:
metrics_out = (
    pd.concat([metrics_tbl, human_metrics_tbl], ignore_index=True)
    .sort_values(['Species', 'Molecule', 'Dose'])
    .rename(columns={
        'Dose':      'Dose (mg/kg)',
        'Obs_Cmax':  'Obs Cmax (µg/mL)',
        'Pred_Cmax': 'Pred Cmax (µg/mL)',
        'FE_Cmax':   'Fold Error Cmax',
        'Obs_AUC':   'Obs AUC (µg·h/mL)',
        'Pred_AUC':  'Pred AUC (µg·h/mL)',
        'FE_AUC':    'Fold Error AUC'
    })
)
for col in metrics_out.select_dtypes(include='number').columns:
    metrics_out[col] = metrics_out[col].round(2)

out_path = os.path.join(FIGURES_DIR, 'PK_Metrics_ObsPred_py.xlsx')
metrics_out.to_excel(out_path, index=False, sheet_name='PK_Metrics')
print(f'Saved: {out_path}')
metrics_out

## 10 — Quick reference: R → Python translations

| R code | Python equivalent | What it does |
|--------|------------------|--------------|
| `read.csv(...)` | `pd.read_csv(...)` | Read CSV file |
| `dplyr::filter(df, x > 0)` | `df.query('x > 0')` | Row filter |
| `dplyr::mutate(df, y = x*2)` | `df.assign(y=lambda d: d.x*2)` | Add/transform column |
| `dplyr::group_by(...) %>% summarise(...)` | `df.groupby(...).agg(...)` | Grouped aggregation |
| `dplyr::left_join(a, b, by='key')` | `a.merge(b, on='key', how='left')` | Table join |
| `tidyr::pivot_wider(...)` | `df.unstack()` / `df.pivot_table(...)` | Wide reshape |
| `data.table::fread(f)` | `pd.read_csv(f)` | Fast CSV read |
| `quantile(x, 0.05)` | `np.quantile(x, 0.05)` | Percentile |
| `ggplot() + geom_line()` | `ax.plot()` | Line plot |
| `geom_ribbon(ymin, ymax)` | `ax.fill_between(y1, y2)` | Shaded band |
| `scale_y_log10()` | `ax.set_yscale('log')` | Log y-axis |
| `coord_cartesian(xlim=c(0,672))` | `ax.set_xlim(0, 672)` | Clip x-axis |
| `facet_wrap(~ Species)` | `subplots(1, n_species)` | Side-by-side panels |
| `kableExtra` coloured table | `df.style.applymap(...)` | Coloured HTML table |
| `openxlsx::saveWorkbook()` | `df.to_excel(...)` | Write Excel file |